In [ ]:
import pandas as pd

df = pd.read_csv('ex2.csv')
df.head(5)

In [ ]:
print('列ラベル：',df.columns)
print('行列サイズ',df.shape)
print('targetの種類とその数：',df['target'].unique(),', ',df['target'].unique().size)
print('target列のデータ個数の集計')
df['target'].value_counts()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
features = ['x0','x1','x2','x3']

# 相関図をプロットする
sns.pairplot(df[features + ["target"]], hue="target")
plt.show()
plt.close

In [ ]:
#print('欠損値の確認')
#print(df.isnull())
print('\nanyメソッドで列単位で欠損値の有無を確認')
print(df.isnull().any(axis=0))
print('\n各列の欠損値の数を調べる')
print(df.isnull().sum())

In [ ]:
# 欠損値を含む行または列の削除
df2 = df.dropna(how='any',axis=0)
print('\n各列の欠損値の数を調べる')
print(df2.isnull().sum())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
features = ['x0','x1','x2','x3']
# df_melt = df.melt(id_vars='target', value_vars=features, var_name='feature', value_name='value')
df_melt = df.melt(value_vars=features, var_name='feature', value_name='value')

# 箱ひげ図を描画
plt.figure(figsize=(10,6))
# sns.boxplot(data=df_melt, x='feature', y='value', hue='target')
sns.boxplot(data=df_melt, x='feature', y='value')
plt.title('Dataset Boxplot')
plt.xlabel('Feature')
plt.ylabel('Value')
plt.legend(title='target')
plt.show()
plt.close

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
features = ['x0','x1','x2','x3']
df_melt = df.melt(id_vars='target', value_vars=features, var_name='feature', value_name='value')
# df_melt = df.melt(value_vars=features, var_name='feature', value_name='value')

# 箱ひげ図を描画
plt.figure(figsize=(10,6))
sns.boxplot(data=df_melt, x='feature', y='value', hue='target')
# sns.boxplot(data=df_melt, x='feature', y='value')
plt.title('Dataset Boxplot')
plt.xlabel('Feature')
plt.ylabel('Value')
plt.legend(title='target')
plt.show()
plt.close

In [ ]:
print('各列の平均値を求める')
print(df.mean(numeric_only=True))
print('各列の中央値を求める')
print(df.median(numeric_only=True))
print('各列の標準偏差を求める')
print(df.std(numeric_only=True))

In [ ]:
# 欠損値をその列の中央値で穴埋めする
colmean = df.median(numeric_only=True)
print(colmean)
df3 = df.fillna(colmean)
print(df3.head(5))

In [ ]:
print('\n各列の欠損値の数を調べる')
print(df3.isnull().sum())

In [ ]:
# 相関図をプロットして元のデータと大きな違いがないかを確認する
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
features = ['x0','x1','x2','x3']

sns.pairplot(df3[features + ["target"]], hue="target")
plt.show()
plt.close

In [ ]:
# 特徴量と正解データを変数に代入
xcol = ['x0', 'x1', 'x2', 'x3']
x = df3[xcol]
t = df3['target']
x,t

In [ ]:
# ホールドアウト法により訓練データとテストデータに分割する
from sklearn.model_selection import train_test_split

# x_train, y_train：学習に利用する訓練データ
# x_test, y_test：検証に利用するテストデータ
# test_size　: 検証に利用する割合、注意：整数値を指定するとテストデータ件数とみなされる
x_train, x_test, y_train, y_test = train_test_split(x,t,test_size=0.2, random_state=0)
print("学習用訓練データの特徴量の行列サイズ：",x_train.shape)
print(x_train.head(3), "\n")
print("検証用テストデータの特徴量の行列サイズ：",x_test.shape)
print(x_test.head(3),"\n")
print("学習用訓練データの正解の行列サイズ：",y_train.shape)
print(y_train.head(3), "\n")
print("検証用テストデータの正解の行列サイズ：",y_test.shape)
print(y_test.head(3),"\n")

In [ ]:
# モジュールのインポート
from sklearn import tree
# モデルの作成
# DecisionTreeClassifier(criterion,　splitter)
#  criterion : gini=ジニ不純度(既定), entropy=情報利得), log_loss=ログ損失(分類の確率的評価))
#　splitter ： best=最良の分割を探索, random=ランダムに候補を選び、その中で最良を選択（高速化向け）
# model = tree.DecisionTreeClassifier(max_depth=2, random_state=0)
model = tree.DecisionTreeClassifier(max_depth=3, random_state=0)
# model = tree.DecisionTreeClassifier(splitter='best')

#　ホールドアウト法により分割した8割の訓練データで学習
model.fit(x_train,y_train)

# モデルの保存
import pickle
with open('ex_model.pkl','wb') as f:
    pickle.dump(model,f)

#　ホールドアウト法により分割した2割のテストデータで正解率を計算
model.score(x_test,y_test)# 

In [ ]:
# 分岐条件の列を表示する
print(model.tree_.feature)
# 分岐条件の閾値を含む配列を表示する
print(model.tree_.threshold)


In [ ]:
# リーフ(葉)に到達したデータの数を返す
print("各ノードに到達した数",model.tree_.weighted_n_node_samples)
print("targetの種類",model.classes_)
idx = [2,4,5,6]
for i in idx:
    print("ノード番号",i,"に到達した割合",model.tree_.value[i]*model.tree_.weighted_n_node_samples[i])



In [ ]:
# plot_tree関数で決定木を描画する
x_train.columns = [['x0', 'x1', 'x2', 'x3']]

from sklearn.tree import plot_tree
plot_tree(model, feature_names=x_train.columns,filled=True)

In [ ]:
print(x.columns)

# 保存したモデルの読み込み
import pickle

with open('ex_model.pkl', 'rb') as f:
    model2 = pickle.load(f)

# 新しいデータのTarget値予測
new_data = [[1.56,0.23,-1.1,-2.8]]
new_data_df = pd.DataFrame(new_data, columns=x.columns)

model2.predict(new_data_df)